# Applied AI — DTW Timestamp Refinement\n\nUse this **after Sony SampleID** has suggested a source region and derivative region.\n\nThis notebook refines the derivative timestamp using librosa subsequence DTW. It does not need Sony's checkpoint.

In [ ]:
# 1) Clone the project and install lightweight dependencies\n!rm -rf /content/projects\n!git clone -q https://github.com/manzaa-glitch/projects.git /content/projects\n!pip -q install librosa pandas scipy soundfile\n\nimport sys\nsys.path.insert(0, '/content/projects/dtw_refinement')\nfrom refine_alignment import refine_candidate\nprint('DTW refiner ready.')

In [ ]:
# 2) Upload ORIGINAL / SOURCE song\nfrom google.colab import files\nprint('Choose the original/source song:')\nup = files.upload()\nsource_path = next(iter(up.keys()))\n

In [ ]:
# 3) Upload SAMPLED / DERIVATIVE song\nprint('Choose the sampled/derivative song:')\nup = files.upload()\nderivative_path = next(iter(up.keys()))\n

In [ ]:
# 4) Paste one Sony candidate here (seconds)\nSOURCE_START = 42.5\nSOURCE_END = 47.5\nDERIVATIVE_HINT_START = 70.0\nDERIVATIVE_HINT_END = 75.0\nSEARCH_PADDING = 8.0\nMAX_PITCH_SHIFT = 6\n\nresult, path = refine_candidate(\n    source_path,\n    derivative_path,\n    source_start_s=SOURCE_START,\n    source_end_s=SOURCE_END,\n    derivative_hint_start_s=DERIVATIVE_HINT_START,\n    derivative_hint_end_s=DERIVATIVE_HINT_END,\n    search_padding_s=SEARCH_PADDING,\n    max_pitch_shift_semitones=MAX_PITCH_SHIFT,\n)\n\nresult.to_dict()

In [ ]:
# 5) Show a cleaner summary and download CSV\nimport pandas as pd\ndf = pd.DataFrame([result.to_dict()])\ndisplay(df)\ndf.to_csv('dtw_refinement.csv', index=False)\nfiles.download('dtw_refinement.csv')

### Interpretation\n\n- **Refined derivative start/end** = where DTW thinks the supplied source excerpt best fits inside the padded derivative search region.\n- **Best pitch shift** = rough transposition that gave the lowest DTW cost.\n- **Mean DTW cost** = relative alignment cost; lower is better, but it is not a probability.\n- **Playback duration ratio** = derivative matched duration / source excerpt duration.\n\nThis is an experiment. Validate the output against known human timestamps before using it in Nature of Use documentation.